# Unitree G1 — Flat Terrain Training
Train con robot đi trên mặt phẳng bằng MuJoCo Playground + Brax PPO trên Kaggle Dual T4.

In [ ]:
# ============================================================
# Cell 1: Kiểm tra GPU
# ============================================================
import subprocess

subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"])

0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB


CompletedProcess(args=['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'], returncode=0)

In [2]:
# ============================================================
# Cell 2: Clone repo & cd vào project
# ============================================================
import os, subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/Bipedal-Locomotion")

if not PROJECT_DIR.exists():
    subprocess.check_call([
        "git", "clone",
        "--branch", "M2",
        "https://github.com/TotallyNotMinh/Bipedal-Locomotion.git",
        str(PROJECT_DIR)
    ])
    print("[✓] Cloned repo")
else:
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "pull"])
    print("[✓] Repo up to date")

os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())

Already up to date.
[✓] Repo up to date
Working dir: /kaggle/working/Bipedal-Locomotion


In [3]:
# ============================================================
# Cell 3: Cài đặt dependencies
# ============================================================
import sys, subprocess

pkgs = [
    "brax",
    "flax",
    "optax",
    "orbax-checkpoint",
    "ml_collections",
    "pyyaml",
    "imageio",
    "imageio-ffmpeg",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade"] + pkgs)

# MuJoCo Playground (v0.2.0 có G1JoystickFlatTerrain)
try:
    import mujoco_playground
    print("[✓] mujoco_playground already installed")
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade", "--no-deps",
        "git+https://github.com/google-deepmind/mujoco_playground.git@v0.2.0"
    ])
    print("[✓] mujoco_playground installed")

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
[✓] mujoco_playground already installed


In [4]:
# ============================================================
# Cell 4: Kiểm tra JAX & import
# ============================================================
import os

# Phải set TRƯỚC KHI import mujoco để EGL được dùng cho rendering
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import jax
devices = jax.devices()
print(f"[✓] JAX {jax.__version__} — {len(devices)} device(s)")
for d in devices:
    print(f"    {d}")

import mujoco_playground
print(f"[✓] mujoco_playground {getattr(mujoco_playground, '__version__', '?')} imported OK")
print(f"[✓] MUJOCO_GL={os.environ.get('MUJOCO_GL','(not set)')}")


[✓] JAX 0.11.1 — 2 device(s)
    cuda:0
    cuda:1
[✓] mujoco_playground ? imported OK
[✓] MUJOCO_GL=egl


In [5]:
# ============================================================
# Cell 5: TRAIN M2 (capacity-matched MLP) trên mặt phẳng
# Env: default G1JoystickFlatTerrain của Playground (không override)
# PPO: hyperparameter chính thức của Playground cho G1 (200M steps)
# ============================================================
import importlib.util
import os
import sys
import yaml

PROJECT_DIR = "/kaggle/working/Bipedal-Locomotion"
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

spec = importlib.util.spec_from_file_location("train_flat", f"{PROJECT_DIR}/scripts/train_flat.py")
train_flat = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_flat)

with open("configs/ppo_flat.yaml") as f:
    cfg = yaml.safe_load(f)

print("Training timesteps:", f"{cfg['ppo']['num_timesteps']:,}")
print("Model:", cfg["experiment"].get("model_id"), "| policy:", cfg["network"]["policy_hidden_layer_sizes"])
meta = train_flat.train(cfg, output_dir=cfg["experiment"].get("output_dir", "results/flat"))
print("\n=== DONE ===")
print("Checkpoint:", meta["checkpoint_dir"])
print("Total time:", meta["total_wall_time_s"], "s")

Training timesteps: 50,000,000
Model: None | policy: [512, 256, 128]
[+] JAX devices (2): ['cuda:0', 'cuda:1']
[+] Output: /kaggle/working/Bipedal-Locomotion/results/flat/g1_flat_20261125_092503
[+] obs_size={'privileged_state': (216,), 'state': (103,)}  action_size=29
[+] Starting PPO training...
  step=         0  elapsed=   282s  sps=         0  eval_reward= -3.522  eval_len=  54.8
  step= 2,785,280  elapsed=   732s  sps=     3,803  eval_reward= -1.890  eval_len=  47.9
  step= 5,570,560  elapsed=   990s  sps=     5,629  eval_reward= -1.531  eval_len=  49.6
  step= 8,355,840  elapsed=  1154s  sps=     7,241  eval_reward= -1.375  eval_len=  56.6
  step=11,141,120  elapsed=  1319s  sps=     8,449  eval_reward= -1.336  eval_len=  61.4
  step=13,926,400  elapsed=  1483s  sps=     9,390  eval_reward= -1.276  eval_len=  67.5
  step=16,711,680  elapsed=  1647s  sps=    10,144  eval_reward= -1.079  eval_len=  78.4
  step=19,496,960  elapsed=  1812s  sps=    10,761  eval_reward= -0.877  eval_

In [6]:
# ============================================================
# Cell 6 (tuỳ chọn): Xem metrics sau khi train xong
# ============================================================
import glob, json

run_dirs = sorted(glob.glob("results/flat/g1_flat_*"))
if not run_dirs:
    print("Chưa có kết quả train")
else:
    latest = run_dirs[-1]
    rows = []
    with open(f"{latest}/metrics.jsonl") as f:
        for line in f:
            rows.append(json.loads(line))

    print(f"Run: {latest}")
    print(f"{'step':>12}  {'eval_reward':>12}  {'eval_len':>10}  {'wall_time_s':>12}")
    for r in rows:
        print(
            f"{r.get('step', 0):>12,}  "
            f"{r.get('eval/episode_reward', float('nan')):>12.3f}  "
            f"{r.get('eval/avg_episode_length', float('nan')):>10.1f}  "
            f"{r.get('wall_time_s', 0):>12.0f}"
        )

Run: results/flat/g1_flat_20261125_092503
        step   eval_reward    eval_len   wall_time_s
           0        -3.522        54.8           282
   2,785,280        -1.890        47.9           732
   5,570,560        -1.531        49.6           990
   8,355,840        -1.375        56.6          1154
  11,141,120        -1.336        61.4          1319
  13,926,400        -1.276        67.5          1483
  16,711,680        -1.079        78.4          1647
  19,496,960        -0.877        95.5          1812
  22,282,240        -0.621       111.4          1976
  25,067,520        -0.567       110.2          2141
  27,852,800        -0.127       133.5          2305
  30,638,080         0.615       179.9          2469
  33,423,360        -0.067       132.4          2634
  36,208,640         0.153       144.0          2798
  38,993,920         0.731       168.6          2963
  41,779,200         1.200       201.5          3128
  44,564,480         1.764       214.4          3292
  47

In [ ]:
import glob
import subprocess
import sys
from base64 import b64encode
from pathlib import Path

from IPython.display import HTML

# Tìm checkpoint mới nhất
run_dirs = sorted(glob.glob("results/flat/g1_flat_*"))
assert run_dirs, "Chưa có checkpoint!"
latest_run = Path(run_dirs[-1]).resolve()
ckpt_root = latest_run / "checkpoints"
step_dirs = sorted(
    [d for d in ckpt_root.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name),
)
ckpt_path = str((step_dirs[-1] if step_dirs else ckpt_root).resolve())
video_path = "/kaggle/working/g1_flat_demo.mp4"

print(f"[+] Checkpoint: {ckpt_path}")
print("[+] Chạy render_demo.py trong subprocess...")

result = subprocess.run(
    [
        sys.executable, "scripts/render_demo.py",
        "--checkpoint", ckpt_path,
        "--out", video_path,
        "--steps", "500",
        "--command", "0.5", "0.0", "0.0",   # vx vy wz cố định cho demo
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("=== STDERR ===")
    print(result.stderr)
    raise RuntimeError(f"render_demo.py failed (code {result.returncode})")

print(f"[✓] Video: {video_path}")
mp4 = open(video_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML(
    f'<video width="480" height="360" controls autoplay loop>'
    f'<source src="{data_url}" type="video/mp4"></video>'
))


: 